# Install

In [87]:
%pip install geopandas shapely requests pyyaml pandas numpy

Note: you may need to restart the kernel to use updated packages.


# Setup

In [88]:
import pandas as pd
import numpy as np
import yaml
import geopandas as gpd
from shapely.geometry import Point
import re

# Load mmc data

In [89]:
path_to_file = "data/mmc-10.yaml"

In [90]:
with open(path_to_file, "r", encoding="utf-8") as file:
    raw_data = yaml.safe_load(file)
    
df = pd.DataFrame(raw_data)

In [91]:
df

,_id,url,topics,authors,date,figures,id,keywords,lead,mention,paragraphs,title,n_comments,gpt_keywords
0,673d34cb91440eaabc59f136,https://www.rtvslo.si/svet/vojna-v-ukrajini/mi...,svet,[B. V.],2024-11-19T15:30:02,[{'caption': 'Vojna v Ukrajini traja že 1000 d...,727977,"[Vojna, Sankcije, Rusija, Invazija, Ukrajina]",V vojni je bilo po podatkih ZN-a ubitih 12.164...,"[Dmitrij Peskov, Volodimir Zelenski, Vladimir ...","[""Ukrajina se ne bo nikoli podredila okupatorj...",Mineva 1000 dni od začetka invazije v Ukrajini,16.0,NaN
1,64cfe08650a608b5567fb4d2,https://www.rtvslo.si/slovenija/golob-rekordna...,slovenija,"[A. S., A. K. K.]",2023-08-04T13:15:42,"[{'caption': 'Marjan Šarec, Robert Golob in Sr...",677096,"[Robert Golob, Medsebojna pomoč, Predsednica d...",Današnja ujma je povzročila verjetno največjo ...,"[Marjan Šarec, Boštjan Poklukar, Srečko Šesta...","[""Po sobotni seji vlade bomo vložili novelo za...",Golob: Rekordna škoda zaradi ujm v samostojni ...,491.0,"[ujma, naravne nesreče, pomoč, premier, zakon,..."
2,686476be7500bfba23052f59,https://www.rtvslo.si/gospodarstvo/pogovori-me...,gospodarstvo,[T. L. Š.],2025-07-01T17:55:32,[{'caption': 'Kot so povedali v premierjevem k...,750667,"[Dialog, Zaprtje poslovalnic, Pogonska goriva,...",Pogovori med vlado in družbo Petrol se bodo po...,"[Robertom Golobom, Sašem Bergerjem, Aleksander...",[Današnji sestanek med Robertom Golobom in Saš...,Pogovori med vlado in Petrolom se bodo nadalje...,NaN,NaN
3,646b0b4ae7b5bc23d6763670,https://www.rtvslo.si/zabava-in-slog/znani/ana...,zabava-in-slog,[K. S.],2023-05-09T12:28:00,[{'caption': 'Bastian in Ana sta par od leta 2...,667491,"[Ana Ivanović, Bastian Schweinsteiger, Rojstvo]",Ana Ivanović in Bastian Schweinsteiger sta se ...,[],"[35-letna nekdanja srbska teniška zvezdnica, k...",Ana Ivanović in Bastian Schweinsteiger še tret...,4.0,"[Ana Ivanović, Bastian Schweinsteiger, otrok, ..."
4,66c7d13a11584f6db9b92754,https://www.rtvslo.si/gospodarstvo/cisti-dobic...,gospodarstvo,[G. K.],2024-08-22T17:47:24,[{'caption': 'Dobiček Luke Koper je bil v prve...,718765,"[Poslovni izid, Investicijski ciklus, Pretovor...",Luka Koper je v drugem letošnjem četrtletju na...,[],[V Luki Koper so nerevidirano polletno poročil...,Čisti dobiček Luke Koper ob polletju višji od ...,1.0,NaN
5,6672c87b1282e10945bb8252,https://www.rtvslo.si/sport/preostali-sporti/j...,sport,"[S. J., D. S. M.]",2024-06-19T12:22:21,[{'caption': 'Plavalka ravenskega Fužinarja Ja...,712298,"[plavanje, evropsko prvenstvo, Janja Šegel]",Na evropskem prvenstvu v plavanju smo videli d...,[Štafeta sedma],"[Šegel, ki bo v Beogradu v finalu nastopila že...","Janja Šegel verjame, da se bo na 200 m prosto ...",0.0,NaN
6,6645f57cd7b7a00dc0ed0bd9,https://www.rtvslo.si/zabava-in-slog/popkultur...,zabava-in-slog,[Ž. E. Č.],2024-05-16T09:17:27,"[{'caption': '""Modna revija Victoria's Secret ...",708416,"[Victoria's Secret, modna revija, vrnitev]",Po večletnem premoru se bodo jeseni supermanek...,"[Adriana Lima, Naomi Campbell, Bella Hadid, Ta...",[Victoria's Secret bo po več letih končno spet...,Vrača se legendarna modna revija Victoria's Se...,4.0,"[supermanekenke, krila, modna revija, Victoria..."
7,6467cd4d7db98a9226d45c8a,https://www.rtvslo.si/znanost-in-tehnologija/s...,znanost-in-tehnologija,[G. C.],2023-04-13T14:31:00,[{'img': 'https://img.rtvcdn.si/_up/upload/202...,664721,"[vesolje, ESA, MGRT, strategija]",Gospodarsko ministrstvo je predstavilo osnutek...,"[Matevž Frangež, Tanja Permozer, Gordon Campbell]","["" Slovenija je majhna na Zemlji, a želi posta...","""Slovenija je majhna na Zemlji, a želi postati...",13.0,"[vesolje, Slovenija, vesoljska industrija, ves..."
8,660685909bf5764acd04d86c,https://www.rtvslo.si/slovenija/premier-in-pos...,slovenija,[La. Da.],2024-03-13T11:28:00,"[{'caption': 'Foto: BoBo', 'img': 'https://img...",701404,"[Gibanje Svoboda, Robert Golob, poslanci]",Vodja poslanske skupine Gibanja Svoboda Borut ...,"[Boruta Sajovica, Petro Škofic, Mateja Arčona,...",[Poslans

# Load city locations

In [92]:
# Za določitev v kateri NUTS regiji je občina
url = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_60M_2024_4326_LEVL_3.geojson"
gdf = gpd.read_file(url)
slovenia_regions = gdf[gdf['NUTS_ID'].str.startswith('SI', na=False)]

def get_slovenia_region(row):
    try:
        # Create the point
        point = Point(row['lng'], row['lat'])
        
        # Iterate through regions
        for _, region in slovenia_regions.iterrows():
            if region['geometry'].contains(point):
                return pd.Series([region['NUTS_NAME'], region['NUTS_ID']])
    except Exception:
        pass

    return pd.Series([None, None])

def slovensko_sklanjanje(row):
    ime = row["naselje"]
    if not isinstance(ime, str) or not ime:
        return pd.Series([None, None])

    # Pomožna funkcija za obdelavo posamezne besede
    def sklanjaj_besedo(beseda):
        # 1. Množinska imena (Abitanti, v Abitantih)
        if beseda.endswith('i'):
            return beseda + 'v', beseda + 'h'
        
        # 2. Ženska imena (Ljubljana, v Ljubljani)
        if beseda.endswith('a'):
            osnova = beseda[:-1]
            return osnova + 'e', osnova + 'i'
        
        # 3. Srednji spol (Velenje, Celje, Trebelno)
        if beseda.endswith('e') or (beseda.endswith('o') and len(beseda) > 3):
            osnova = beseda[:-1]
            # Preverimo prevoj (C, Č, Ž, Š, J) - v rodilniku ni vpliva, v mestniku pa
            return osnova + 'a', osnova + 'u'

        # 4. Moški spol (Maribor, Mokronog, Gradec)
        # Reševanje izpadajočega polglasnika (enostaven algoritem za -ec)
        osnova = beseda
        if beseda.endswith('ec'):
            osnova = beseda[:-2] + 'c'
        elif beseda.endswith('el') and beseda != 'Velenje': # npr. Angel -> Angla
            osnova = beseda[:-2] + 'l'
            
        return osnova + 'a', osnova + 'u'

    # Razbijemo na dele (upoštevamo presledke in vezaje)
    # Regex razbije "Mokronog-Trebelno" na ['Mokronog', '-', 'Trebelno']
    deli = re.split(r'(\s+|-)', ime)
    
    rodilnik_deli = []
    mestnik_deli = []
    
    for del_imena in deli:
        if del_imena.strip() == '' or del_imena == '-':
            rodilnik_deli.append(del_imena)
            mestnik_deli.append(del_imena)
        else:
            r, m = sklanjaj_besedo(del_imena)
            rodilnik_deli.append(r)
            mestnik_deli.append(m)
            
    return pd.Series(["".join(rodilnik_deli), "".join(mestnik_deli)])

In [93]:
# Za združitev občin z naselji
naselje_obcina_file = "data/naselje-obcina.csv"
obcina_geoloc_file = "data/obcina-geoloc.csv"

naselje_obcina = pd.read_csv(naselje_obcina_file, sep=";") 
obcina_geoloc = pd.read_csv(obcina_geoloc_file)

df_naselje_cord = pd.merge(
    naselje_obcina, 
    obcina_geoloc[['city', 'lat', 'lng']], 
    left_on='občina', 
    right_on='city', 
    how='left'
)
df_naselje_cord = df_naselje_cord.drop(columns=['city'])
df_naselje_cord = df_naselje_cord.drop(columns=['občina'])
df_naselje_cord = df_naselje_cord.dropna()

df_naselje_cord[["rodilnik", "mestnik"]] = df_naselje_cord.apply(slovensko_sklanjanje, axis=1)

# Poda regijo kordinati
df_naselje_cord[['region_name', 'region_id']] = df_naselje_cord.apply(get_slovenia_region, axis=1)

# Dropa random občine, ki so pre blizu borderja, da jim kordinati zajbavajo
df_naselje_cord = df_naselje_cord.dropna()

# Da so imena na začetku
cols = ["naselje", "rodilnik", "mestnik", "lat", "lng", "region_name", "region_id"]
df_naselje_cord = df_naselje_cord[cols]

In [94]:
# Odstranbe

# Krka ker podjetje
bannedBesede = ["tabor", "kot", "krog", "konec", "svet", "vrh", "ravni", "liga", "svetu", "vrata", 
				"koncu", "KRKA", "ter", "vrhu", "okrog", "rob", "sedlo", "vir", "Jeruzalem", "nemci",
				"meja", "ravno", "železnice", "križ", "plače", "pogled", "starše", "hudo", "srednje", 
				"ladja", "površju", "globoko", "ceste", "gradnja", "naredi", "Kralji", "gola", "anže", 
				"občina", "jesen", "prazniki", "bele", "vojska", "vrt"] 
for beseda in bannedBesede:
	df_naselje_cord = df_naselje_cord.query(
    'naselje.str.lower() != @beseda.lower() and '
    'rodilnik.str.lower() != @beseda.lower() and '
    'mestnik.str.lower() != @beseda.lower()'
	)

In [95]:
# Tu lahko preveriš, če je beseda res odstranjena
beseda = "rob"
df_naselje_cord.query(
    'naselje.str.lower() == @beseda.lower() or '
    'rodilnik.str.lower() == @beseda.lower() or '
    'mestnik.str.lower() == @beseda.lower()'
	)

,naselje,rodilnik,mestnik,lat,lng,region_name,region_id


In [96]:
print(df_naselje_cord.head())
print(df_naselje_cord.count())

     naselje    rodilnik     mestnik      lat      lng            region_name  \
0   Abitanti   Abitantiv   Abitantih  45.5500  13.7333          Obalno-kraška   
1    Adamovo     Adamova     Adamovu  45.8363  14.6377      Osrednjeslovenska   
3   Adlešiči   Adlešičiv   Adlešičih  45.5711  15.1889  Jugovzhodna Slovenija   
4  Adrijanci  Adrijanciv  Adrijancih  46.8050  16.2172               Pomurska   
5       Ajba        Ajbe        Ajbi  46.0880  13.6347                Goriška   

  region_id  
0     SI044  
1     SI041  
3     SI037  
4     SI031  
5     SI043  
naselje        4884
rodilnik       4884
mestnik        4884
lat            4884
lng            4884
region_name    4884
region_id      4884
dtype: int64


In [97]:
df_naselje_cord.to_csv("./processed_data/naselja.csv")